# Lesson 4.4 — Converting Between a DataFrame and a 2D Array

**Objectives**
- Turn a pandas `DataFrame` into a 2D NumPy array with `.to_numpy()` and understand what is lost
- Turn a 2D NumPy array back into a `DataFrame` with column names and an index
- Do a calculation in NumPy and put the result back into a `DataFrame`
- Know when to use a `DataFrame` and when to use an array

### Why convert at all?

A `DataFrame` and a 2D array are both tables of rows and columns. The difference is **labels**:

| 2D NumPy array                    | pandas DataFrame                         |
|-----------------------------------|------------------------------------------|
| Rows and columns are just numbers | Rows have an index, columns have names   |
| One `dtype` for the whole array   | Each column can have its own `dtype`     |
| Fast math, `axis`, broadcasting   | Filtering, groupby, joins, reading files |
| What scikit-learn wants as input  | What you load, clean and explore with    |

In practice you load and clean data as a `DataFrame`, hand a 2D array to NumPy or scikit-learn
for the number crunching, and then wrap the result back into a `DataFrame` so the labels come back.
This lesson is that round trip.

In [ ]:
import numpy as np
import pandas as pd

## 1. A small DataFrame to work with

In [ ]:
sales = pd.DataFrame(
    {
        "jan": [120, 80, 200],
        "feb": [135, 95, 210],
        "mar": [150, 70, 190],
    },
    index=["store_a", "store_b", "store_c"],
)
sales

## 2. DataFrame → 2D array: `.to_numpy()`

`.to_numpy()` returns the underlying 2D array. Row order and column order are kept, but the index and column names are **dropped**:

In [ ]:
arr = sales.to_numpy()

print(arr)
print(type(arr))
print("shape:", arr.shape)   # (rows, columns) - same as sales.shape
print("dtype:", arr.dtype)

> You may also see `df.values` in older code. It does the same thing; `.to_numpy()` is the
> recommended spelling today.

Once it is an array, everything from Lessons 4.2 and 4.3 applies: `[row, col]` indexing, `axis`, masks:

In [ ]:
print(arr[0, 1])          # store_a, feb
print(arr[:, 0])          # the jan column
print(arr.sum(axis=1))    # total per store/row
print(arr[arr > 150])     # all values above 150

Compare with the labeled version in pandas. Same answers, but you ask by **name** instead of position:

In [ ]:
print(sales.loc["store_a", "feb"])
print(sales["jan"].to_numpy())
print(sales.sum(axis=1))

## 3. Watch the `dtype` when columns are mixed

An array has a single `dtype`. If the DataFrame mixes ints and floats you get floats. If it
mixes numbers and text you get `object`, which is slow and loses all NumPy math. Convert only
the numeric columns you actually need.

In [ ]:
customers = pd.DataFrame(
    {
        "name": ["Ana", "Ben", "Cho"],
        "age": [34, 28, 45],
        "spend": [120.5, 80.0, 200.25],
    }
)

everything = customers.to_numpy()
print(everything)
print("dtype with text column:", everything.dtype)

In [ ]:
numeric = customers[["age", "spend"]].to_numpy()
print(numeric)
print("dtype numeric only    :", numeric.dtype)

`select_dtypes` picks the numeric columns for you when there are many:

In [ ]:
numeric_cols = customers.select_dtypes(include="number")
print(numeric_cols.columns.tolist())
print(numeric_cols.to_numpy())

## 4. 2D array → DataFrame: `pd.DataFrame(array, columns=..., index=...)`

Going the other way is a single call. Without labels you get numbered columns `0, 1, 2`:

In [ ]:
grid = np.arange(12).reshape(3, 4)

pd.DataFrame(grid)

Pass `columns=` (and optionally `index=`) to add labels back. The lengths must match the array's shape:

In [ ]:
pd.DataFrame(
    grid,
    columns=["q1", "q2", "q3", "q4"],
    index=["north", "south", "west"],
)

A 1D array becomes a single column (or a `Series`):

In [ ]:
one_d = np.array([10, 20, 30])

print(pd.Series(one_d, name="value"))
pd.DataFrame(one_d, columns=["value"])

## 5. The round trip: DataFrame → array → math → DataFrame

This is the pattern you will use constantly. Reuse the original DataFrame's `columns` and
`index` so the labels line up again.

In [ ]:
arr = sales.to_numpy()

# 1) math in NumPy: what share of each store's total came from each month?
share = arr / arr.sum(axis=1, keepdims=True)   # keepdims keeps a (3, 1) column so broadcasting works row by row

# 2) wrap it back up with the same labels
share_df = pd.DataFrame(share, columns=sales.columns, index=sales.index).round(3)
share_df

Another classic: **min-max scaling** each column to the range 0–1, which machine learning models often need:

In [ ]:
scaled = (arr - arr.min(axis=0)) / (arr.max(axis=0) - arr.min(axis=0))

pd.DataFrame(scaled, columns=sales.columns, index=sales.index).round(2)

You can also write an array straight into new DataFrame columns, as long as the number of rows matches:

In [ ]:
result = sales.copy()
result["total"] = arr.sum(axis=1)
result["best_month"] = sales.columns[arr.argmax(axis=1)]
result

## 6. Views vs copies, again

In Lesson 4.2 you saw that a NumPy slice is a *view*. `.to_numpy()` on a DataFrame **may** also
return a view of pandas' internal data, so writing into the array could silently change the
DataFrame. Whenever you plan to modify the array, ask for a copy explicitly:

In [ ]:
safe = sales.to_numpy(copy=True)
safe[0, 0] = -999

print(safe[0, 0])
print(sales.loc["store_a", "jan"])   # unchanged

## 7. With the real CO2 dataset

Let's do the round trip on a few numeric columns for Canada from the course dataset.

In [ ]:
co2_emission = pd.read_csv("../../data/owid-co2-data.csv", sep=",")

canada = (
    co2_emission.loc[
        (co2_emission["country"] == "Canada") & (co2_emission["year"] >= 2015),
        ["year", "coal_co2", "oil_co2", "gas_co2"],
    ]
    .set_index("year")   # year becomes the row label, so only real numbers are left in the table
)
canada

In [ ]:
fuels = canada.to_numpy()

print("shape:", fuels.shape)
print("dtype:", fuels.dtype)

Now NumPy math with `axis`, then back into a labeled table. What percentage of each year's fossil CO2 came from each fuel?

In [ ]:
percent = fuels / fuels.sum(axis=1, keepdims=True) * 100

percent_df = pd.DataFrame(percent, columns=canada.columns, index=canada.index).round(1)
percent_df

Because the labels came back, pandas conveniences work on the result right away:

In [ ]:
print(percent_df.idxmax(axis=1))   # the biggest fuel each year
print(percent_df.mean().round(1))  # average share per fuel

### When to stay in pandas

Notice that `canada.div(canada.sum(axis=1), axis=0)` gives the same percentages without ever
leaving pandas. For simple math, stay in pandas: it keeps the labels for you. Reach for
`.to_numpy()` when a library needs an array (scikit-learn in a later module), when you need
NumPy-only functions, or when you want the raw speed of a large numeric block.

## Try it yourself

Use this DataFrame of exam scores:

```python
scores = pd.DataFrame(
    {"math": [78, 92, 65, 88], "science": [85, 79, 70, 95], "english": [90, 84, 72, 80]},
    index=["ana", "ben", "cho", "dev"],
)
```

1. Convert `scores` to a 2D array and print its `shape` and `dtype`
2. Using the array, compute each student's average score (one number per row)
3. Subtract each subject's mean from the array (hint: `axis=0`), then wrap the result in a new `DataFrame` with the same columns and index
4. Using the CO2 data: build a 2D array of `co2` and `co2_per_capita` for `United States` from 2015 onward, scale each column to 0–1 with min-max scaling, and return it as a `DataFrame` indexed by `year`

In [ ]:
scores = pd.DataFrame(
    {"math": [78, 92, 65, 88], "science": [85, 79, 70, 95], "english": [90, 84, 72, 80]},
    index=["ana", "ben", "cho", "dev"],
)

# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO

### Solution

In [ ]:
# 1.
scores_arr = scores.to_numpy()
print(scores_arr.shape, scores_arr.dtype)

In [ ]:
# 2.
print(scores_arr.mean(axis=1))

In [ ]:
# 3.
centered = scores_arr - scores_arr.mean(axis=0)
pd.DataFrame(centered, columns=scores.columns, index=scores.index)

In [ ]:
# 4.
us = co2_emission.loc[
    (co2_emission["country"] == "United States") & (co2_emission["year"] >= 2015),
    ["year", "co2", "co2_per_capita"],
].set_index("year")

us_arr = us.to_numpy()
us_scaled = (us_arr - us_arr.min(axis=0)) / (us_arr.max(axis=0) - us_arr.min(axis=0))

pd.DataFrame(us_scaled, columns=us.columns, index=us.index).round(3)